In [6]:
import os

# 查看当前工作目录
current_directory = os.getcwd()
print(current_directory)

/home/cquptyyz


In [7]:
# 切换到目标目录
os.chdir('/home/cquptyyz/Code_MODMA/Self_adaptve_gnn_multiple_domain')

# 确认已切换
print(os.getcwd())

/home/cquptyyz/Code_MODMA/Self_adaptve_gnn_multiple_domain


In [69]:
from load_data import load_data_for_model
# data_type = 'MODMA_EEG'
data_type = 'TD_BRAIN_EEG_TYT_2s'
# input_folder = '/home/cquptyyz/EEGdata/MODMA_elecdtrode_2s'
input_folder = '/home/cquptyyz/EEGdata/TD_BRAIN_2preprocessed_data.npz'
eegs, labels = load_data_for_model(data_type, input_folder)

In [70]:
eegs.shape

(10800, 26, 1000)

In [44]:
# import numpy as np
# import torch
# from blocks_yyz import extract_frequency_bands
# datas = []
# for i in range(eegs.shape[0]):
#     eeg = eegs[i]
#     psd, de, se = extract_frequency_bands(eeg, fs=250, nperseg=50, noverlap=25)
#     feature = np.stack((A, B, C), axis=0) # psd = feature[0].shape, de = feature[1].shape, se = feature[2].shape
#     datas.append(feature)
    
import numpy as np
import torch
from blocks_yyz import extract_frequency_bands
datas = []
for i in range(eegs.shape[0]):
    eeg = eegs[i]
    psd, de, se = extract_frequency_bands(eeg, fs=250, nperseg=50, noverlap=25)
    feature = np.stack((psd, de, se), axis=0)
    datas.append(feature)

In [45]:
datas = np.array(datas)

In [52]:
datas.shape

(8012, 3, 128, 5)

In [67]:
yyz = datas[:, 0, :, :]
yyz.shape

(8012, 128, 5)

In [53]:
labels.shape

(8012,)

In [54]:
from sklearn.model_selection import train_test_split
X_train, X_test, Y_train, Y_test = train_test_split(datas, labels, test_size=0.1, random_state=0)

In [55]:
from sklearn.model_selection import StratifiedKFold
def cross_validate_sequential_split(x_data, y_labels, kfold):
    train_indices = {}
    eval_indices = {}
    skf = StratifiedKFold(n_splits=kfold, shuffle=False)  # random_state=42
    i = 0
    for train_idx, eval_idx in skf.split(x_data, y_labels):
        train_indices.update({i: train_idx})
        eval_indices.update({i: eval_idx})
        i += 1
    return train_indices, eval_indices
train_indices, eval_indices = cross_validate_sequential_split(X_train, Y_train, kfold=10)

In [56]:
### ============ Prepare data for every fold ============
train_idx = train_indices.get(1)  # 取出第k折的训练数据
eval_idx = eval_indices.get(1)  # 取出第k折的验证数据

In [57]:
import numpy as np
import torch
def split_xdata(eeg_data, train_idx, eval_idx):
    x_train = np.copy(eeg_data[train_idx, :, :])
    x_eval = np.copy(eeg_data[eval_idx, :, :])
    x_train = torch.from_numpy(x_train).to(torch.float32)
    x_eval = torch.from_numpy(x_eval).to(torch.float32)
    return x_train, x_eval

def split_ydata(y_true, train_idx, eval_idx):
    y_train = np.copy(y_true[train_idx])
    y_eval = np.copy(y_true[eval_idx])
    y_train = torch.from_numpy(y_train).to(torch.int64)
    y_eval = torch.from_numpy(y_eval).to(torch.int64)
    return y_train, y_eval

x_train, x_eval = split_xdata(X_train, train_idx, eval_idx)
y_train, y_eval = split_ydata(Y_train, train_idx, eval_idx)

In [58]:
from torch.utils.data import TensorDataset, DataLoader
batch_size = 64
train_data = TensorDataset(x_train, y_train)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=False)
eval_data = TensorDataset(x_eval, y_eval)
eval_loader = DataLoader(eval_data, batch_size=batch_size, shuffle=False, num_workers=0, drop_last=False)

In [68]:
for inputs, target in train_loader:
    print("Inputs:", inputs.shape)
    print("Target:", target.shape)
    # 如果只想查看一次，可以加上 break
    break

Inputs: torch.Size([64, 3, 128, 5])
Target: torch.Size([64])


In [66]:
datas.shape[3]

5

In [2]:
os.chdir('/home/cquptyyz/Code_MODMA/Self_adaptve_gnn_multiple_domain')
print(os.getcwd())

/home/cquptyyz/Code_MODMA/Self_adaptve_gnn_multiple_domain


In [3]:
from load_data import load_subject_independent_data_for_model
from blocks_yyz import Adopt_Gnn_PSD_DE_SE, extract_frequency_bands
import numpy as np
### ========================= Load data ===============================================###
data_type = 'MODMA_EEG'
input_folder = '/home/cquptyyz/EEGdata/MODMA_elecdtrode_2s'
selected_folders = {'02010002', '02010004', '02020008', '02020010'}
test_eegs, test_labels, train_eegs, train_labels = load_subject_independent_data_for_model(input_folder, selected_folders)
test_datas = []
train_datas = []
fs = 250
nperseg = 50
noverlap = 25
for i in range(test_eegs.shape[0]):
    eeg = test_eegs[i]
    psd, de, se = extract_frequency_bands(eeg, fs=fs, nperseg=nperseg, noverlap=noverlap)
    feature = np.stack((psd, de, se), axis=0)
    test_datas.append(feature) # (8012, 3, 128, 5) [batch_size, 3, eeg_channels, features]
test_datas = np.array(test_datas)
for i in range(train_eegs.shape[0]):
    eeg = train_eegs[i]
    psd, de, se = extract_frequency_bands(eeg, fs=fs, nperseg=nperseg, noverlap=noverlap)
    feature = np.stack((psd, de, se), axis=0)
    train_datas.append(feature)  # (8012, 3, 128, 5) [batch_size, 3, eeg_channels, features]
train_datas = np.array(train_datas)

file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02010005, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02030021, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02020018, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02030009, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02020016, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02020026, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02030018, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02010012, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02030004, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02020025, npy_num: 167
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02030014, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02010013, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02010002, npy_num: 150
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_2s/02030019, npy_n

In [4]:
train_datas.shape

(7410, 3, 128, 5)

In [5]:
train_labels.shape

(7410,)

In [6]:
test_datas.shape

(602, 3, 128, 5)

In [7]:
test_labels.shape

(602,)

In [9]:
import statistics
import os
import re
import sys
from datetime import datetime
import numpy as np
import torch
import torch.optim as optim
from torch import nn
from blocks_yyz import Adopt_Gnn_PSD_DE_SE, extract_frequency_bands
import matplotlib.pyplot as plt
# from tqdm import tqdm
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score, precision_score, accuracy_score
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from load_data import load_data_for_model, load_subject_independent_data_for_model
### ========================= Initialization parameters ===============================================###
def create_dataloader(data, labels, shuffle, batch_size=64):
    tensor_data = torch.from_numpy(data).to(torch.float32)
    tensor_labels = torch.from_numpy(labels).to(torch.int64)
    dataset = TensorDataset(tensor_data, tensor_labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataset, dataloader
dim = train_datas.shape[3]  # 特征维度
eeg_channels = train_datas.shape[2]
subgraph_size = 20  # k = 20
alpha_PSD = 3
alpha_DE = 3
alpha_SE = 3
beta_PSD = 0.05
beta_DE = 0.05
beta_SE = 0.05
gcn_depth = 2
layers = 3
num_classes = 2  # 几分类
train_epochs = 100  # 15 10 5 20 2 10
batch_size = 64  # 32
kfolds = 10  # 10 5
test_size = 0.1

### ========================= Split data & Cross validate ==============================================
# 打乱并分割数据
kf = KFold(n_splits=kfolds, shuffle=True, random_state=42)
folds = list(kf.split(train_datas))

### ========================= train model ===============================================###
eval_kfold_acc = []
eval_kfold_recall = []
eval_kfold_specificity = []
eval_kfold_precision = []
eval_kfold_f1 = []

In [10]:
for kfold, (train_idx, eval_idx) in enumerate(folds):

    ### ============ Initialization Performance ============
    best_acc = 0
    best_recall = 0
    best_specificity = 0
    best_f1 = 0
    best_epoch = -1
    best_kfold = -1

    ### ============ Prepare data for every fold ============
    x_train = train_datas[train_idx], y_train = train_labels[train_idx]
    x_eval = train_datas[eval_idx], y_eval = train_labels[eval_idx]

    ### ============ Initialization model ============
    # model
    model = Adopt_Gnn_PSD_DE_SE(dim, eeg_channels, subgraph_size, device, alpha_PSD, alpha_DE, alpha_SE, beta_PSD, beta_DE, beta_SE, gcn_depth, layers, num_classes=num_classes).to(device)
    # 优化器
    optimizer = optim.Adam(model.parameters(), lr=1e-4, betas=(0.5, 0.999), weight_decay=1e-4) # lr=1e-3
    # 损失函数
    criterion = nn.CrossEntropyLoss(reduction='sum').to(device) # reduction='sum'表示计算的loss是求和的

    # 设置每一折的最佳模型保存位置
    folder_bestmodel_file = os.path.join(save_model_file + f"/{kfold}fold_model.pth")

    # 保存训练以及验证集的损失
    train_losses = []
    eval_losses = []
    break

ValueError: too many values to unpack (expected 2)

In [3]:
input_folder = '/home/cquptyyz/EEGdata/MODMA_elecdtrode_10s'

In [4]:
inputlist = os.listdir(input_folder)

In [6]:
len(inputlist)

53

In [8]:
inputlist.sort()
inputlist

['02010002',
 '02010004',
 '02010005',
 '02010006',
 '02010008',
 '02010010',
 '02010011',
 '02010012',
 '02010013',
 '02010015',
 '02010016',
 '02010018',
 '02010019',
 '02010021',
 '02010022',
 '02010023',
 '02010024',
 '02010025',
 '02010026',
 '02010028',
 '02010030',
 '02010033',
 '02010034',
 '02010036',
 '02020008',
 '02020010',
 '02020013',
 '02020014',
 '02020015',
 '02020016',
 '02020018',
 '02020019',
 '02020020',
 '02020021',
 '02020022',
 '02020023',
 '02020025',
 '02020026',
 '02020027',
 '02020029',
 '02030002',
 '02030003',
 '02030004',
 '02030005',
 '02030006',
 '02030007',
 '02030009',
 '02030014',
 '02030017',
 '02030018',
 '02030019',
 '02030020',
 '02030021']

In [15]:
import random
# 设置随机种子以保证结果可复现
random.seed(42)

# 打乱列表以进行随机划分
random.shuffle(inputlist)

# 计算每一份的大小
fold_size = len(inputlist) // 10

# 划分为10等分
folds = [inputlist[i * fold_size:(i + 1) * fold_size] for i in range(10)]

# 如果有剩余的样本，分配到各个fold中
remainder = len(inputlist) % 10
for i in range(remainder):
    folds[i].append(inputlist[10 * fold_size + i])

# 打印每一折的内容
for i, fold in enumerate(folds):
    print(f"Fold {i+1}: {fold}")

Fold 1: ['02010015', '02010036', '02020010', '02010006', '02010033', '02010004']
Fold 2: ['02020027', '02010024', '02020029', '02010028', '02010018', '02010012']
Fold 3: ['02030018', '02020008', '02020021', '02020016', '02020019', '02030002']
Fold 4: ['02030006', '02010008', '02020015', '02010016', '02020013']
Fold 5: ['02020025', '02010002', '02030007', '02010026', '02030004']
Fold 6: ['02030020', '02020023', '02030017', '02020018', '02010030']
Fold 7: ['02010034', '02010019', '02030021', '02020020', '02030009']
Fold 8: ['02010021', '02030003', '02030019', '02010005', '02020014']
Fold 9: ['02020026', '02010010', '02020022', '02030005', '02010011']
Fold 10: ['02010013', '02010022', '02010023', '02010025', '02030014']


In [10]:
from sklearn.model_selection import train_test_split, KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)
folds = list(kf.split(inputlist))

In [11]:
for kfold, yyz in enumerate(folds):
    print(yyz)
    break

(array([ 0,  1,  2,  3,  4,  6,  7,  8,  9, 10, 11, 13, 14, 15, 16, 17, 18,
       20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 39, 40, 42, 44, 45, 46, 48, 49, 50, 51, 52]), array([ 5, 12, 19, 41, 43, 47]))


In [16]:
import os
from sklearn.model_selection import KFold

# 输入文件夹路径
input_folder = '/home/cquptyyz/EEGdata/MODMA_elecdtrode_10s'
inputlist = os.listdir(input_folder)
inputlist.sort()

# 设置 KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# 打印每一折的内容
for i, (train_index, test_index) in enumerate(kf.split(inputlist)):
    train_fold = [inputlist[idx] for idx in train_index]
    test_fold = [inputlist[idx] for idx in test_index]
    print(f"Fold {i+1}:")
    print(f"  Train: {train_fold}")
    print(f"  Test: {test_fold}")

Fold 1:
  Train: ['02010002', '02010004', '02010005', '02010006', '02010008', '02010011', '02010012', '02010013', '02010015', '02010016', '02010018', '02010021', '02010022', '02010023', '02010024', '02010025', '02010026', '02010030', '02010033', '02010034', '02010036', '02020008', '02020010', '02020013', '02020014', '02020015', '02020016', '02020018', '02020019', '02020020', '02020021', '02020022', '02020023', '02020025', '02020026', '02020027', '02020029', '02030002', '02030004', '02030006', '02030007', '02030009', '02030017', '02030018', '02030019', '02030020', '02030021']
  Test: ['02010010', '02010019', '02010028', '02030003', '02030005', '02030014']
Fold 2:
  Train: ['02010002', '02010004', '02010005', '02010008', '02010010', '02010011', '02010012', '02010015', '02010016', '02010018', '02010019', '02010022', '02010023', '02010024', '02010026', '02010028', '02010030', '02010033', '02010034', '02010036', '02020008', '02020010', '02020013', '02020014', '02020015', '02020016', '020200

In [20]:
from load_data import load_data_for_model, load_subject_independent_data_for_model, load_designated_subjects_data_from_MODMA


In [21]:
train_eegs, train_labels = load_designated_subjects_data_from_MODMA(input_folder, train_fold)

file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010002, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010004, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010005, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010006, npy_num: 31
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010008, npy_num: 33
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010010, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010011, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010012, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010013, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010015, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010016, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010018, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010019, npy_num: 30
file: /home/cquptyyz/EEGdata/MODMA_elecdtrode_10s/02010021, npy_

In [23]:
train_eegs.shape

(1447, 128, 2501)

In [24]:
train_labels.shape

(1447,)

In [2]:
report_file = f"test_report.md"

In [3]:
report_file

'test_report.md'

In [8]:
import os
report_file = os.path.join(current_directory + report_file)
report_file

'/home/cquptyyztest_report.md'

In [10]:
from datetime import datetime
import numpy as np
import torch
import torch.optim as optim
from torch import nn
from blocks_yyz import Adopt_Gnn_PSD_DE_SE, extract_frequency_bands
import matplotlib.pyplot as plt
# from tqdm import tqdm
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, f1_score, recall_score, precision_score, accuracy_score
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from load_data import load_data_for_model, load_subject_independent_data_for_model, load_designated_subjects_data_from_MODMA

input_folder = '/home/cquptyyz/EEGdata/MODMA_elecdtrode_10s'
input_list = os.listdir(input_folder)
input_list.sort()
# 设置外层KFold
folds_out = 10
folds_in = 10
kf_out = KFold(n_splits=folds_out, shuffle=True, random_state=42)
### ========================= 开始外循环（10*10交叉验证）===================================
for i_out, (train_index, test_index) in enumerate(kf_out.split(input_list)):
    train_fold = [input_list[idx] for idx in train_index] # 外层交叉中划分为训练集的被试id列表
    test_fold = [input_list[idx] for idx in test_index] # 外层交叉中划分为测练集的被试id列表
    print(f"Out_Fold:{i_out+1}:")
    print(f"  Train subjects: {train_fold}")
    print(f"  Test subjects : {test_fold}")

    # 设置内层KFold
    kf_in = KFold(n_splits=folds_in, shuffle=True, random_state=18)

    eval_kfold_acc = []
    eval_kfold_recall = []
    eval_kfold_specificity = []
    eval_kfold_precision = []
    eval_kfold_f1 = []

    ### ========================= 开始内循环（10*10交叉验证）===============================
    for i_in, (train_index_in, valid_index_in) in enumerate(kf_in.split(train_fold)):
        train_fold_in = [train_fold[idx] for idx in train_index_in] # 内层交叉中划分为训练集的被试id列表
        valid_fold_in = [train_fold[idx] for idx in valid_index_in] # 内层交叉中划分为验证集的被试id列表
        print(f"Out_Fold:{i_out + 1} In_Fold:{i_in + 1}:")
        print(f"  Train subjects: {train_fold_in}")
        print(f"  Valid subjects : {valid_fold_in}")

Out_Fold:1:
  Train subjects: ['02010002', '02010004', '02010005', '02010006', '02010008', '02010011', '02010012', '02010013', '02010015', '02010016', '02010018', '02010021', '02010022', '02010023', '02010024', '02010025', '02010026', '02010030', '02010033', '02010034', '02010036', '02020008', '02020010', '02020013', '02020014', '02020015', '02020016', '02020018', '02020019', '02020020', '02020021', '02020022', '02020023', '02020025', '02020026', '02020027', '02020029', '02030002', '02030004', '02030006', '02030007', '02030009', '02030017', '02030018', '02030019', '02030020', '02030021']
  Test subjects : ['02010010', '02010019', '02010028', '02030003', '02030005', '02030014']
Out_Fold:1 In_Fold:1:
  Train subjects: ['02010002', '02010004', '02010005', '02010006', '02010008', '02010011', '02010012', '02010013', '02010015', '02010016', '02010018', '02010021', '02010023', '02010024', '02010025', '02010026', '02010030', '02010033', '02010034', '02010036', '02020008', '02020010', '02020014